# DataFrame Baseline — Naive Bayes Classifier

**Unoptimised implementation using the PySpark DataFrame / SQL API.**

This notebook is the baseline reference — no optimisations applied.
It uses Python UDFs wherever possible (UDFs are opaque to Spark's Catalyst
optimizer and involve Python serialisation overhead) and does not call
`.persist()` or `.repartition()`.
See `naive_bayes_df_optimized.ipynb` for the improved version.

The algorithm is identical to the RDD version: the same `compute_log_probs()`,
`predict()`, and `evaluate()` functions from `core/naive_bayes.py` are used,
ensuring that probability outputs match the RDD implementation exactly.

In [1]:
# Cell 1 — Imports and SparkSession setup
import time
import math
import sys
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

sys.path.insert(0, os.path.abspath('..'))

from core.naive_bayes import compute_log_probs, evaluate
from data.loader import load_car_dataframe, FEATURE_COLS, LABEL_COL

# predict is defined locally so cloudpickle serialises it by value (bytecode).
# Functions imported from external modules are serialised by reference — workers
# would need to import core/, which requires the module to be present on every
# worker node. Defining predict here avoids that dependency entirely.
def predict(log_prob_table, test_point):
    best_class = None
    best_score = float('-inf')
    for class_label in log_prob_table['classes']:
        score = log_prob_table['log_class_probs'][class_label]
        for feat_idx, value in enumerate(test_point):
            key = f'feat_{feat_idx}_{value}_{class_label}'
            if key in log_prob_table['log_feature_probs']:
                score += log_prob_table['log_feature_probs'][key]
            else:
                score += log_prob_table['fallback_log_probs'].get(
                    (feat_idx, class_label), math.log(1e-10)
                )
        if score > best_score:
            best_score = score
            best_class = class_label
    return best_class

# TODO (Dataproc): Remove .master() — the cluster provides its own master URL.
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('NaiveBayes-DF-Baseline')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')
print('SparkSession ready.')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 14:06:10 WARN Utils: Your hostname, MacBook-Pro-de-Lucca.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.184 instead (on interface en0)
26/04/02 14:06:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 14:06:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession ready.


In [2]:
# Cell 2 — Load data
#
# TODO: Replace None with your dataset path.
#   Local      : '/Users/you/data/car.data'
#   Dataproc   : 'gs://your-bucket/car.data'
# Leave as None to run immediately with the 5-row dummy dataset.
DATA_PATH = '/Users/luccagazotto/Documents/École Polytechnique/DSAIB/Database Management/final_project/Database-Management---Naive-Bayes/data/car.csv'

train_df, test_df = load_car_dataframe(spark, filepath=DATA_PATH)

print(f'Train rows : {train_df.count()}')
print(f'Test rows  : {test_df.count()}')
print('Schema:')
train_df.printSchema()

# Show class distribution to verify the data loaded correctly.
# Schema at this point: [buying, maint, doors, persons, lug_boot, safety, label]
print('Class distribution in training data:')
train_df.groupBy(LABEL_COL).count().orderBy('count', ascending=False).show()

Train rows : 1431
Test rows  : 297
Schema:
root
 |-- buying: string (nullable = true)
 |-- maint: string (nullable = true)
 |-- doors: string (nullable = true)
 |-- persons: string (nullable = true)
 |-- lug_boot: string (nullable = true)
 |-- safety: string (nullable = true)
 |-- label: string (nullable = true)

Class distribution in training data:
+-----+-----+
|label|count|
+-----+-----+
|unacc| 1008|
|  acc|  314|
| good|   58|
|vgood|   51|
+-----+-----+



In [3]:
# Cell 3 — Training phase
#
# BASELINE BEHAVIOUR — deliberately unoptimised:
#   - A Python UDF constructs the string key for each (feature, value, class) combo.
#     UDFs are opaque to Catalyst: Spark cannot inspect, merge, or optimise them.
#     Every UDF call involves Python serialisation overhead.
#   - No .persist() on intermediate DataFrames.
#   - No .repartition() — partitioning is whatever the loader produced.

t_train_start = time.time()

# Count how many training rows belong to each class.
# Schema before: [buying, maint, doors, persons, lug_boot, safety, label]
# Schema after : [label, class_count]
class_counts_df = train_df.groupBy(LABEL_COL).agg(F.count('*').alias('class_count'))

# UDF: build the key string 'feat_{idx}_{value}_{class}' for each feature observation.
# A UDF is used here (BASELINE) because we want an explicit Python function to
# construct the string — matching the mapper convention from Zheng (2014).
# In the optimised version, this is replaced by F.concat() which Catalyst can optimise.
make_key_udf = F.udf(
    lambda feat_idx, value, label: f'feat_{feat_idx}_{value}_{label}',
    StringType()
)

# Unpivot: each feature column becomes a separate row with a key string.
# We build one DataFrame per feature, then union them into one tall DataFrame.
# This is the DataFrame equivalent of flatMap in the RDD version.
# Schema of each key_df: [key]
# Example key value: 'feat_0_low_unacc'
key_dfs = []
for feat_idx, col_name in enumerate(FEATURE_COLS):
    key_df = train_df.select(
        make_key_udf(F.lit(feat_idx), F.col(col_name), F.col(LABEL_COL)).alias('key')
    )
    key_dfs.append(key_df)

# Union all feature key DataFrames into one tall DataFrame.
# Schema: [key]   e.g. 'feat_0_low_unacc'
all_keys_df = key_dfs[0]
for df in key_dfs[1:]:
    all_keys_df = all_keys_df.union(df)

# Count occurrences of each key — this is the REDUCE step.
# Schema before: [key]
# Schema after : [key, count]
feature_counts_df = all_keys_df.groupBy('key').agg(F.count('*').alias('count'))

train_time = time.time() - t_train_start
print(f'[TIMING] Training: {train_time:.4f}s')
print(f'Unique feature keys: {feature_counts_df.count()}')
feature_counts_df.show(5)

[TIMING] Training: 0.1016s
Unique feature keys: 69
+------------------+-----+
|               key|count|
+------------------+-----+
|  feat_0_vhigh_acc|   62|
|    feat_0_low_acc|   68|
|    feat_0_med_acc|   94|
|  feat_0_med_unacc|  211|
|feat_0_vhigh_unacc|  298|
+------------------+-----+
only showing top 5 rows


In [4]:
# Cell 4 — Probability computation
#
# We collect both DataFrames to the driver as Python dicts, then call
# compute_log_probs() from core/naive_bayes.py — the same function used
# by the RDD implementation, guaranteeing identical probability outputs.

class_counts_dict   = {row[LABEL_COL]: row['class_count'] for row in class_counts_df.collect()}
feature_counts_dict = {row['key']: row['count'] for row in feature_counts_df.collect()}
class_totals        = class_counts_dict

log_prob_table = compute_log_probs(class_counts_dict, feature_counts_dict, class_totals)

print(f"Classes           : {log_prob_table['classes']}")
print(f"Log class probs   : {log_prob_table['log_class_probs']}")
print(f"Sample feat probs : {list(log_prob_table['log_feature_probs'].items())[:3]}")

Classes           : ['unacc', 'acc', 'vgood', 'good']
Log class probs   : {'unacc': -0.35220510784011255, 'acc': -1.5163474893680884, 'vgood': -3.317676409612294, 'good': -3.191382684288002}
Sample feat probs : [('feat_0_vhigh_acc', -1.6189166563886441), ('feat_0_low_acc', -1.5279448781829175), ('feat_0_med_acc', -1.208174491179636)]


In [5]:
# Cell 5 — Prediction
#
# BASELINE BEHAVIOUR:
#   A Python UDF applies predict() to each test row.
#   The UDF captures log_prob_table in its closure — Spark serialises
#   the dict with every task (no broadcast). This is the unoptimised behaviour.
#
# A UDF is required here (in both baseline and optimised) because predict()
# involves Python procedural logic (iterating over classes, argmax) that
# cannot be expressed as a Spark SQL expression.
predict_udf = F.udf(
    lambda *features: predict(log_prob_table, list(features)),
    StringType()
)

t_pred_start = time.time()

# Apply the UDF to every test row, adding a 'prediction' column.
# Schema before: [buying, maint, doors, persons, lug_boot, safety, label]
# Schema after : [buying, maint, doors, persons, lug_boot, safety, label, prediction]
prediction_df = test_df.withColumn(
    'prediction',
    predict_udf(*[F.col(c) for c in FEATURE_COLS])
)

# Force evaluation by collecting the label and prediction columns.
results = prediction_df.select(LABEL_COL, 'prediction').collect()

pred_time = time.time() - t_pred_start
print(f'[TIMING] Prediction: {pred_time:.4f}s')
print(f'Sample: {results[:5]}')

[TIMING] Prediction: 0.0713s
Sample: [Row(label='unacc', prediction='unacc'), Row(label='unacc', prediction='unacc'), Row(label='unacc', prediction='unacc'), Row(label='unacc', prediction='unacc'), Row(label='unacc', prediction='unacc')]


In [ ]:
# Cell 6 — Evaluation

true_labels = [row[LABEL_COL] for row in results]
pred_labels = [row['prediction'] for row in results]

accuracy = evaluate(pred_labels, true_labels)

print(f'Accuracy     : {accuracy:.4f} ({accuracy * 100:.2f}%)')
print(f'Train time   : {train_time:.4f}s')
print(f'Predict time : {pred_time:.4f}s')
print(f'Total time   : {train_time + pred_time:.4f}s')

# TODO: Expected accuracy on the full car evaluation dataset is ~87.39%
# (Zheng 2014). With the 5-row dummy dataset accuracy is not meaningful.
# Replace DATA_PATH with the real file path before interpreting results.

Accuracy     : 0.8552 (85.52%)
Train time   : 0.1016s
Predict time : 0.0713s
Total time   : 0.1729s


26/04/02 14:43:57 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 952717 ms exceeds timeout 120000 ms
26/04/02 14:43:57 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/02 14:43:59 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$